# Example: Using `FGClean`

This notebook provides an example of how to use the `FGClean` class (in the `fgclean.py` module of `hdfgclean`) to run the foreground (FG) cleaning procedure described in [MacInnis et. al. (2026)](https://arxiv.org/abs/XXXX.XXXXX) on a set of maps.

In particular, although you may use any maps compatible with [pixell](https://pixell.readthedocs.io) with `FGClean`, we will use the same set of $2^\circ \times 2^\circ$ degree maps generated by [hdsims](https://github.com/CMB-HD/hdsims) that are used in the `hdfgclean_example_2x2.ipynb` notebook. We will use `hdsims` to save the maps; then, from that point forward, we will treat them as a general set of maps.

The notebook is broken in to three main steps:
1. Define the arguments passed to `FGClean` and save them in a configuration `.yaml` file
2. Run the FG cleaning with `run_fgclean.py` (outside of the notebook)
3. Load in the FG-cleaned maps and the catalogs of detected point sources and clusters

Below, we import the python packages/modules needed to run this notebook:

In [ ]:
import os
import numpy as np
import matplotlib 
import matplotlib.pyplot as plt
%matplotlib inline
from pixell import enmap
from hdsims import hdsims, siminfo as si, utils, fgcatalogs, maps, plots
from hdfgclean import fgclean, fgmaps, fgfilters, hdfgclean_utils

In the cell below, you **must** provide the path to an `output_dir` where the output from FG cleaning will be saved.
- Note: you cannot use the same `output_dir` as a previous FG-cleaning run (this is not the case when using `HDFGClean`, which creates a new sub-directory for each new set of maps); in general, we suggest that you provide a new, empty directory for your `output_dir`


You may also change the `sims_dir`, where the maps passed to `FGClean` will be saved (within a sub-directory of that directory); by default, this is the same as the `output_dir`.

You will need about 6 GB of space to save the simulations, and about 3 GB to save the FG cleaning output.

In [ ]:
output_dir = 
sims_dir = output_dir

If you have moved (or made a copy of) this notebook into a different directory, provide the path to the `hdfgclean` repository (i.e., the directory that contains the readme file) below; otherwise, you can leave this cell unchanged:

In [ ]:
hdfgclean_repo_dir = None

---

# Before FG cleaning: set up and save the configuration file

## Download the simulations

Here, we will download the simulations (generated with `hdsims`) used to run the FG cleaning in this example. They are 0.04 arcminute-resolution maps of the beam- and pixel-window-convolved sum of the lensed CMB, tSZ, kSZ, CIB, and radio sources, plus instrumental noise, at 90, 148, 219, and 277 GHz. We also need a second set of simulations, which are a different realization of the maps described above, to quantify the noise in the matched filter calculation (these are smaller versions of the ones described in the `hdfgclean` "[README](https://github.com/CMB-HD/hdfgclean/blob/main/README.md#before-using-hdfgclean)" file; also see the relevant section in the `run_hdfgclean.ipynb` notebook).

In the following cell, we will print out intructions to download the necessary simulations from [github](https://github.com/CMB-HD/sim_files_for_example_notebooks).

In [ ]:
hdfgclean_utils.print_instructions_to_download_2x2hdsims(sims_dir)

After following the instructions above, you must run the cell below to make sure the files were saved correctly; if they weren't, follow the instructions that are printed out:

In [ ]:
hd_sims_saved = hdfgclean_utils.hdsims_for_example_are_saved(sims_dir)
if not hd_sims_saved:
    hdfgclean_utils.print_instructions_to_download_2x2hdsims(sims_dir)
    print(f"\nThen, re-run this notebook cell.")

## Save the maps that will be passed to `FGClean`

Below, we initialize the `HDSims` class (in the `hdsims.py` module of the `hdsims` package) for the maps we just downloaded:

In [ ]:
freqs = [90, 148, 219, 277] # in GHz
width = 2 # in degrees
height = width
apod_width = 0.2 # in degrees

# sims to be FG-cleaned:
simlib = hdsims.HDSims(sims_dir, freqs=freqs, pol=False,
                       width=width, height=height, apod_width=apod_width)


# sims used to quantify noise in matched filter calculations:
ra_ctr_for_noise = 26 # in degrees; we don't change the dec.
cmb_seed_for_noise = 15
white_noise_seeds = {freq: freq for freq in freqs}
simlib4filt = hdsims.HDSims(sims_dir, freqs=freqs, pol=False,
                            width=width, height=height, apod_width=apod_width,
                            ra_ctr=ra_ctr_for_noise, cmb_seed=cmb_seed_for_noise,
                            noise_seeds=white_noise_seeds)

In [ ]:
# geometry of the maps: includes extra room around edges to apodize
map_shape = simlib.padded_shape
map_wcs = simlib.padded_wcs
map4filt_wcs = simlib4filt.padded_wcs # different wcs b/c different location on sky

We will save these maps in sub-directories of your `sims_dir`; we define the map file names below:

In [ ]:
# dictionaries of file names for maps:
imap_fnames = {}
maps_for_source_filter_fnames = {}
maps_for_cluster_filter_fnames = {}
maps_for_source_mask_filter_fnames = {}

imaps_dir = utils.mkdir(os.path.join(sims_dir, 'maps_for_fgclean_example'))
maps4filt_dir = utils.mkdir(os.path.join(imaps_dir, 'maps_for_filter_calculations'))
maps4filt_path = lambda fname : os.path.join(maps4filt_dir, fname)

for freq in freqs:
    freq_info = f'{freq:03d}GHz'
    # map to be FG cleaned:
    imap_fnames[freq] = os.path.join(imaps_dir, f'{freq_info}_map.fits')
    # maps for matched filter calculations:
    map_info = 'map_for_source_filter'
    maps_for_source_filter_fnames[freq] = maps4filt_path(f'{freq_info}_{map_info}.fits')
    map_info = 'map_for_cluster_filter'
    maps_for_cluster_filter_fnames[freq] = maps4filt_path(f'{freq_info}_{map_info}.fits')
    map_info = 'map_for_source_mask_filter'
    maps_for_source_mask_filter_fnames[freq] = maps4filt_path(f'{freq_info}_{map_info}.fits')

Here we define the remaining arguments that *must* always be passed to `FGClean`, the dictionaries of map file names and the beam full-width at half-maximum (arcminutes) at each frequency:

In [ ]:
# use `hdsims.siminfo` module to get the beam sizes:
beam_fwhms = {freq: si.beam_fwhm[freq] for freq in freqs}

# save apodized, beam- and pixel-window-convolved 
# sims with noise at each frequency:
if hd_sims_saved:
    for freq in freqs:
        # maps being FG-cleaned:
        if not os.path.exists(imap_fnames[freq]):
            sim_map = simlib.get_sim(freq=freq, beam=True, noise=True, 
                                     shape=map_shape, wcs=map_wcs)
            enmap.write_map(imap_fnames[freq], sim_map)

Now we will also save the sets of maps used to quantify the noise in the matched filter calculations. For further details, you may refer to the relevant section in the `run_hdfgclean.ipynb` notebook, or to MacInnis et. al. (2026)

In [ ]:
# save maps used to quantify the noise in the matched filter calculations:

# load in pre-computed catalogs of sources and clusters that were
# detected in this second set of maps:
maps4filt_detected_sources, maps4filt_detected_clusters = hdfgclean_utils.detected_fg_catalogs_for_2x2filter()

# if the maps used in the matched filter calculations have not been saved, 
# then we will need to make a map of clusters that were detected in 
# those maps (in order to get an estimate of the residual clusters after 
# FG cleaning); we define the (positional and keyword) arguments needed 
# to make those maps:
cluster_map_args = [map_shape, map4filt_wcs, maps4filt_detected_clusters,
                    fgfilters.get_default_gauss_cluster_profiles_dict(), freqs]
cluster_map_kwargs = {'convolve_pixwin': True, 'convolve_beam': True, 'beam_fwhms': beam_fwhms}
# here, we make sure that we only calculate the detected cluster map once
# (in Compton y-units), and then use that to get a map at each frequency
detected_cluster_maps4filt = None

if hd_sims_saved:
    for freq in freqs:
        # maps for point source filters: includes white noise + CMB + kSZ + (all) tSZ
        if not os.path.exists(maps_for_source_filter_fnames[freq]):
            map4filt = simlib4filt.get_sim(freq=freq, beam=True, noise=True, 
                                           components=['cmb', 'ksz', 'tsz'],
                                           shape=map_shape, wcs=map4filt_wcs)
            enmap.write_map(maps_for_source_filter_fnames[freq], map4filt)
        # maps for cluster filters: includes white noise + CMB + kSZ + *residual* sources
        if not os.path.exists(maps_for_cluster_filter_fnames[freq]):
            # get the map with all point sources:
            map4filt = simlib4filt.get_sim(freq=freq, beam=True, noise=True, 
                                           components=['cmb', 'ksz', 'cib', 'radio'],
                                           shape=map_shape, wcs=map4filt_wcs)
            # make a map of detected point sources:
            source_catalog = maps4filt_detected_sources[freq]
            sources_to_subtract = maps.make_src_map(map_shape, map4filt_wcs, [source_catalog], freq)
            sources_to_subtract = enmap.apply_window(sources_to_subtract)
            sources_to_subtract = maps.convolve_sim_with_beam(sources_to_subtract, beam_fwhms[freq])
            map4filt -= sources_to_subtract
            enmap.write_map(maps_for_cluster_filter_fnames[freq], map4filt)
        # maps for filter used when making point soure mask:
        # includes  white noise + CMB + kSZ + *residual* tsz clusters
        if not os.path.exists(maps_for_source_mask_filter_fnames[freq]):
            map4filt = simlib4filt.get_sim(freq=freq, beam=True, noise=True, 
                                           components=['cmb', 'ksz', 'tsz'],
                                           shape=map_shape, wcs=map4filt_wcs)
            if detected_cluster_maps4filt is None:
                detected_cluster_maps4filt = fgmaps.make_cluster_sims_from_catalog(*cluster_map_args,
                                                                                   **cluster_map_kwargs)
            map4filt -= detected_cluster_maps4filt[freq]
            enmap.write_map(maps_for_source_mask_filter_fnames[freq], map4filt)

## Save the configuration file

We also need to pass the `map_apod_width` used to apodize the maps. Because we are using a smaller value (0.2 degrees) than the default, we must also change the `patch_apod_width` used to apodize each smaller patch in the original maps (in this case, our maps are already small, so there is only a single patch)

In [ ]:
# also need to pass:
map_apod_width = simlib.map_apod_width
patch_apod_width = map_apod_width # only one patch

# save the config file
config_file = os.path.join(output_dir, 'fgclean_example2x2.yaml')
if not os.path.exists(config_file):
    fgclean.FGClean.save_config(config_file, imap_fnames, beam_fwhms, output_dir=output_dir, overwrite=False, 
                                map_apod_width=map_apod_width,
                                patch_apod_width=patch_apod_width,
                                noise_maps_for_source_filters=maps_for_source_filter_fnames,
                                noise_maps_for_cluster_filters=maps_for_cluster_filter_fnames,
                                noise_maps_for_source_mask_filters=maps_for_source_mask_filter_fnames)

# Instructions to run the FG cleaning

Here we print out instructions to run the FG cleaning with the `run_fgclean` method of `FGClean`, initialized with your configuration file; this will be done by running the `run_fgclean.py` python script provided in the `hdfgclean` github repository.

**Note**: you do not need to use MPI to run the FG cleaning in this example, but if you are running on a cluster, we still recommend that you do **not** run the FG cleaning procedure (the second command printed out below) on the login nodes. 

In [ ]:
hdfgclean_utils.print_fgclean_example_instructions(config_file, hdfgclean_repo_dir=hdfgclean_repo_dir)

# After FG cleaning: results

In the following cells, we initialize the `FGClean` class, and then make sure everything has been saved.

Note that we can pass additional keyword arguments to the `from_config` method. Here, we pass the geometry (`shape` and `wcs`; see the `pixell.enmap` module) of the maps in advance. If we don't do this, `FGClean` will load it a map to get its geometry (stored as a class attribute); this is fine since our maps are small, but you may wish to avoid this if you have larger maps.

In [ ]:
fgcleanlib = fgclean.FGClean.from_config(config_file, map_shape=map_shape, map_wcs=map_wcs)

In [ ]:
maps_are_fgcleaned = fgcleanlib.all_patches_fgcleaned()

---

In the rest of this notebook, we show how the methods of the `FGClean` class can be used after the FG cleaning has been completed. We do not provide a complete description of each method used here; you should always refer to the documentation (i.e., the docstring written under the method definition in the relevant module of `hdfgclean`) of any method before using it (see the note below). For additional information, see the `hdfgclean` [readme](https://github.com/CMB-HD/hdfgclean/blob/main/README.md) file or refer to MacInnis et. al. (2026).

**Note**: we have ensured (when running `run_fgclean.py` in Part 1) that everything we use below has already been saved. In general, the methods of `HDFGClean` will perform whatever calculations are necessary in order to return whatever you have requested. For example, if you have not run the FG cleaning and call `get_fgcleaned_maps`, the full FG cleaning procedure will run in order to return the FG-cleaned maps.

---

Below, we load in the 148 GHz maps before and after FG cleaning; to get the latter map, we use the `get_fgcleaned_maps` method. Note that the maps and catalogs returned by `FGClean` will always include the apodized region (in contrast to `HDFGClean`, which will cut out the inner, un-apodized map region by default).

In [ ]:
freq = 148
map_before = enmap.read_map(imap_fnames[freq])
# get the map after FG cleaning at this frequency:
maps_after = fgcleanlib.get_fgcleaned_maps(freqs=[freq]) 
map_after = maps_after[freq]

In [ ]:
plots.plot_maps([map_before, map_after], 
                labels=[f'{freq} GHz before FG cleaning', f'{freq} GHz after FG cleaning'], 
                grid=False, colorbar_ranges=[-300,300])

We can also load in and plot maps of all point sources and clusters that were subtracted from the 148 GHz map (or, in general, make them from the detected catalogs if they had not been saved):

In [ ]:
subtracted_sources_map = fgcleanlib.get_map_of_subtracted_sources(freq)
subtracted_clusters_map = fgcleanlib.get_map_of_subtracted_clusters(freq)

plots.plot_maps([subtracted_sources_map, subtracted_clusters_map], 
                labels=[f'CIB+Radio sources\nremoved at {freq} GHz', f'tSZ clusters removed at {freq} GHz'], 
                grid=False, colorbar_ranges=[-50, 50])

Next, we load in and print out the 148 GHz detected point source and cluster catalogs. The catalogs are sorted by signal-to-noise ratio (SNR), with the highest-SNR detection in the first row.

Both of these catalogs have columns named `'RADeg`' and `'decDeg'` for the right ascension (R.A.) and declination (dec.), in degrees, of each detected source or cluster, and a column named `'SNR'` for the SNR of the detection.  For clusters, the R.A. and dec. coordinates are the location where the SNR of the detection was at its maximum, which we assume to be the center of the cluster.
- The detected cluster catalogs also have a column named `'y_c'` for the measured Compton-y parameter at the cluster center, and a column named `'template'` for the name of the cluster profile that produced the highest SNR detection for that cluster.
- The detected source catalogs also have a column named `'fluxmJy'` for the measured flux (in mJy) of the source at that frequency, and a column named `'component'` that, when possible, identifies the source as either `'cib'` or `'radio'`.

The catalogs also have other columns that are used for "book-keeping" while the FG cleaning is running; we do not print those out in this notebook for clarity.

In [ ]:
detected_sources = fgcleanlib.get_catalog_of_subtracted_sources(freq=freq)
detected_clusters = fgcleanlib.get_catalog_of_subtracted_clusters()

In [ ]:
print("detected clusters:")
detected_cluster_cols = ['RADeg', 'decDeg', 'y_c', 'SNR', 'template']
fgcatalogs.display_catalog(detected_clusters[detected_cluster_cols])

In [ ]:
print(f"{freq} GHz detected point sources:")
detected_source_cols = ['RADeg', 'decDeg', 'fluxmJy', 'SNR', 'component']
fgcatalogs.display_catalog(detected_sources[detected_source_cols])

You may have noticed that there are sources with negative SNR in the catalog. These are sources that were detected at 90 or 277 GHz, but not at 148 GHz; we assume that they are dim (at 148 GHz) radio or CIB sources, respectively, and remove them from the 148 GHz map using the measured average 90-to-148 radio or 277-to-148 CIB spectral index to calculate their 148 GHz flux. We calculate the SNR at the locations of these sources in the 148 GHz map *before* subtracting them; the SNR at some locations may be negative, e.g., if there is a tSZ cluster at the same location (which has a negative signal at 148 GHz). 

We can look at the measured average spectral indices:

In [ ]:
mean_cib_index, _ = fgcleanlib.get_spectral_index(freq, 'cib')
print(f"Measured average 277-to-148 CIB spectral index = {mean_cib_index:.2f}")

mean_radio_index, _ = fgcleanlib.get_spectral_index(freq, 'radio')
print(f"Measured average 90-to-148 radio spectral index = {mean_radio_index:.2f}")

We can count how many of the 148 GHz detected sources were identified as radio or CIB sources, and how many could not be identified because they were not matched to a 90 or 277 GHz detected source:

In [ ]:
total_num_sources = len(detected_sources)
num_cib = len(detected_sources[detected_sources['component'].eq('cib')])
num_radio = len(detected_sources[detected_sources['component'].eq('radio')])
num_unknown = len(detected_sources[detected_sources['component'].eq('unknown')])

print(f"{total_num_sources} sources were removed from the {freq} GHz map")
print(f"{num_cib} were identified as CIB, and {num_radio} were identified as radio sources; ")
print(f"{num_unknown} could not be identified, because they were not also found at 90 (if applicable) or 277 GHz")

By default, we make a mask to mask out locations where we have over- or under-subtracted a point source or cluster. Below, we will load in the 148 GHz mask and plot it:

In [ ]:
mask = fgcleanlib.get_mask(freq)

In [ ]:
plots.plot_map(mask, cbar_label='', figsize=(3,3), title=f'{freq} GHz mask') # no units for colorbar